# 0r · **이어서 학습** (resume) — GPU 2장 노드

끊긴 학습을 **마지막 체크포인트부터** 이어 돌린다. 처음부터 다시 하지 않는다.

## ⚠️ 예전엔 resume 이 안 걸렸다 (버그)
lerobot 은 `train_config.json` 을 **체크포인트 안**에만 저장한다:

```
<out>/checkpoints/<step>/pretrained_model/train_config.json     ← 여기
<out>/train_config.json                                          ← 이런 파일은 없음
```

우리 코드가 없는 쪽(`<out>/train_config.json`)을 보고 있어서

1. **resume 이 한 번도 안 걸렸다** → 재실행할 때마다 step 0 부터 다시 학습
2. "config 없음 = 빈 디렉토리"로 오판해 **체크포인트째 지울 뻔했다**

지금은 `checkpoints/last` 를 찾아 `--resume=true --config_path=<...>/train_config.json` 을 붙이고,
**체크포인트가 하나라도 있으면 절대 삭제하지 않는다.**

- 이미 150k 끝난 seed 는 **자동 skip** (덮어쓰지 않음)
- 중간(예: 120k)까지 간 seed 는 **거기서 이어감**
- 체크포인트가 아예 없는 크래시 잔재만 정리하고 처음부터


## 0) 전체 스캔 — **어디가 끊겼나** (task × 그룹 전부)
`insertion` / `transfer` 양쪽, ours·acm·baseline·ablation 을 훑어 **미완 학습**을 찾는다.
이걸 보고 아래 `TASK`·`TAGS` 를 정한다.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

GROUPS = {'ours': cf.GROUP_OURS, 'acm': cf.GROUP_ACM,
          'baseline': cf.GROUP_BASELINE, 'ablation': cf.GROUP_ABLATION}

print(f'목표 {cf.STEPS:,} step | seeds {cf.MAIN_SEEDS}')
print()
for task in (cf.MAIN_SIM, cf.SHORT_SIM):          # insertion, transfer
    print(f'===== {task} =====')
    any_run = False
    for gname, tags in GROUPS.items():
        for t in tags:
            steps = {s: cf.v23.last_ckpt_step(cf.v23.train_dir(t, s, task)) for s in cf.MAIN_SEEDS}
            if not any(v for v in steps.values()):
                continue                      # 한 번도 안 돌린 것은 생략
            any_run = True
            cells = []
            for s, v in steps.items():
                if v is None:
                    cells.append(f'seed{s}:-')
                elif v >= cf.STEPS:
                    cells.append(f'seed{s}:완료')
                else:
                    cells.append(f'seed{s}:{v // 1000}k')
            left = [s for s, v in steps.items() if v is None or v < cf.STEPS]
            mark = '완료' if not left else '▶ 이어야 함'
            print(f'  {gname:<9} {t:<12} ' + '  '.join(f'{c:<12}' for c in cells) + f'  {mark}')
    if not any_run:
        print('  (학습 기록 없음)')
    print()


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

# ── 무엇을 이어서 ────────────────────────────────────────────────────────────
TASK  = cf.MAIN_SIM          # ★ 위 스캔 보고 고를 것: cf.MAIN_SIM='insertion' / cf.SHORT_SIM='transfer'
TAGS  = cf.GROUP_OURS        # ['ours'].  cf.GROUP_ACM / cf.GROUP_BASELINE / cf.GROUP_ABLATION
SEEDS = cf.MAIN_SEEDS        # [0,1,2,3] — 끝난 건 알아서 skip
GPUS  = cf.v23.available_gpus()     # 이 노드의 GPU (2장이면 2잡씩 청크)

print('task :', TASK, '| 모델:', TAGS)
print('seeds:', SEEDS, '| GPU:', GPUS, f'({len(GPUS)}장 → {len(GPUS)}잡씩)')
print('목표 :', f'{cf.STEPS:,} step')

## 1) 현재 상태 — 어디까지 갔나
각 seed 의 **마지막 체크포인트 step** 과 남은 step. 여기서 이어질 것만 뽑는다.

In [ ]:
todo = cf.resume_status(TAGS, SEEDS, TASK)

## 2) resume 커맨드 확인 (dry-run)
`--resume=true` 와 `--config_path=.../checkpoints/<step>/pretrained_model/train_config.json` 이
붙는지 본다. 안 붙으면 그 seed 는 체크포인트가 없어 처음부터 도는 것.

In [ ]:
for t, s in todo:
    c = cf.make_train_cmd(t, seed=s, task=TASK, gpu_id=GPUS[0])
    r = [p for p in c.split() if p.startswith(('--resume', '--config_path', '--steps'))]
    step = cf.v23.last_ckpt_step(cf.v23.train_dir(t, s, TASK))
    head = f'{t}/seed{s}'
    print(f"{head:<16} {'resume @' + format(step, ',') if step else '처음부터':<16} {' '.join(r)}")
    print()

## 3) 이어서 학습
- 끝난 seed 는 skip, 나머지는 마지막 체크포인트에서 이어감.
- GPU 2장이면 2잡씩 청크로 (각 청크가 끝날 때까지 대기).
- 또 끊겨도 이 노트북을 다시 돌리면 그 지점부터 이어간다.

In [ ]:
jobs = cf.run_training(TAGS, SEEDS, task=TASK, gpus=GPUS)

## 4) 결과 확인 — 전부 150k 인가

In [ ]:
cf.resume_status(TAGS, SEEDS, TASK)
print()
ok = cf.print_ckpt_status(TAGS, SEEDS, TASK)
print('\n=>', '다음: eval 노트북' if ok else '⚠️ 아직 남았다 — 위 3) 을 다시 실행')

## 참고 — 왜 끊겼나 (재발 방지)
지금까지 학습이 죽은 원인은 전부 환경 쪽이었고 모두 고쳐졌다:

| 증상 | 원인 | 조치 |
|---|---|---|
| `does not contain any parquet file` | 잡 N개가 같은 HF 캐시에 **동시 다운로드** | 학습 전 `prefetch_dataset` 1회 |
| `RuntimeError: 0 active drivers` | **없는 GPU** 지정(2-GPU 노드에 GPU 2,3) | `cf.part()` 가 노드의 GPU 를 자동 배정 + 사전 검증 |
| 재실행이 step 0 부터 | resume 이 안 걸림(위 버그) | `checkpoints/last` 에서 `--resume` |

그래도 노드가 죽으면(preemption 등) 이 노트북을 다시 돌리면 된다 — **10k 마다 체크포인트**라
잃는 건 최대 10k step.
